In [ ]:
import pandas as pd
import numpy as np
import torch
import re
from tqdm import tqdm
import xgboost as xgb
from transformers import AutoModel, AutoTokenizer

Input the full or relative path to the .csv file containing your data.

In [ ]:
path = ''

In [2]:
# load german bert for embeddings
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-german-cased")
model = AutoModel.from_pretrained("dbmdz/bert-base-german-cased")

In [ ]:
# ready data
data = pd.read_csv(path, index_col=0)
data.head()

48105


,context,id,author,date,text,text_tokenised,token_count
0,bt,sp18_8,Andreas Lenz,2016-01-28,Die Herausforderungen nehmen aber zu . Wir hab...,Die Herausforderungen nehmen aber zu Wir habe...,58
1,bt,sp18_10,Jan-Marco Luczak,2016-01-28,In gleicher Weise kritisch wie diese Absenkung...,In gleicher Weise kritisch wie diese Absenkung...,94
2,bt,sp18_13,Marie-Luise Dött,2016-01-28,Meine sehr geehrten Damen und Herren! In den k...,Meine sehr geehrten Damen und Herren In den k...,89
3,bt,sp18_15,Volker Ullrich,2016-01-28,In den letzten Jahren sind aus guten und polit...,In den letzten Jahren sind aus guten und polit...,83
4,bt,sp18_17,Detlef Seif,2016-01-28,"Die Kommission benennt dann Themen, die sie al...",Die Kommission benennt dann Themen die sie al...,92


In [ ]:
# add keyword counts to data
keywords = ['klima', 'erwärmung', 'treibhaus', 'co2', 'kohle', 
                  'energiewende', 'verkehrswende', 'fridays for future',
                  'extinction rebellion']

for key in keywords:
    count_col = key + '_count'
    data[count_col] = data['text'].apply(lambda x: len(re.findall(key, str(x).lower())))

,context,id,author,date,text,text_tokenised,token_count,klima_count,erwärmung_count,treibhaus_count,co2_count,kohle_count,energiewende_count,verkehrswende_count,fridays for future_count,extinction rebellion_count,year
0,bt,sp18_8,Andreas Lenz,2016-01-28,Die Herausforderungen nehmen aber zu . Wir hab...,Die Herausforderungen nehmen aber zu Wir habe...,58,1,0,0,0,0,0,0,0,0,2016
1,bt,sp18_10,Jan-Marco Luczak,2016-01-28,In gleicher Weise kritisch wie diese Absenkung...,In gleicher Weise kritisch wie diese Absenkung...,94,1,0,0,0,0,0,0,0,0,2016
2,bt,sp18_13,Marie-Luise Dött,2016-01-28,Meine sehr geehrten Damen und Herren! In den k...,Meine sehr geehrten Damen und Herren In den k...,89,3,0,0,0,0,0,0,0,0,2016
3,bt,sp18_15,Volker Ullrich,2016-01-28,In den letzten Jahren sind aus guten und polit...,In den letzten Jahren sind aus guten und polit...,83,1,0,0,0,0,0,0,0,0,2016
4,bt,sp18_17,Detlef Seif,2016-01-28,"Die Kommission benennt dann Themen, die sie al...",Die Kommission benennt dann Themen die sie al...,92,1,0,0,0,0,0,0,0,0,2016


The classifier was trained with the year of the text's authorship as a feature. You may wish to add a column 'year' with the respective year or np.NaN values.

In [ ]:
##### STEP 1: Embeddings

# NOTE: THIS CAN BE SKIPPED IF EMBEDDINGS HAVE ALREADY BEEN GENERATED
# In that case proceed to Step 2 and ensure the correct path to the embeddings

chunks = []
chunk_size = 50
data_texts = data['text'].to_list()
chunks = [data_texts[i:i+chunk_size] for i in range(0, len(data_texts), chunk_size)]

all_embeddings = []

with tqdm(total=len(chunks), desc="Embeddings chunks") as pbar:    
    
    for chunk in chunks:
        chunk_embeds = []

        for text in chunk:

            ##### embed texts with german-bert-cased
            inputs = tokenizer(text, padding=True, truncation=True, return_tensors='pt')
            
            with torch.no_grad():
                outputs = model(**inputs)
                embeddings = outputs.last_hidden_state[:, 0, :]

            chunk_embeds.append(embeddings.numpy()[0, :]) # save only first row to keep output 2d

        all_embeddings.extend(chunk_embeds)
        pbar.update(1)

embeddings = np.array(all_embeddings)

# save embeddings
np.save('embeddings.npy', embeddings)

Embeddings chunks: 100%|██████████| 811/811 [6:08:39<00:00, 27.27s/it]  


In [ ]:
##### STEP 2: Create X

embeddings = np.load('embedidngs.npy')  #load embeddings from memory
embeddings_df = pd.DataFrame(embeddings)

# grab features from data
features = ['klima_count','erwärmung_count',
                'treibhaus_count','co2_count','kohle_count',
                'energiewende_count','verkehrswende_count',
                'fridays for future_count','extinction rebellion_count']

# NOTE: If you have the column 'year', add it here, if not, comment out line
features = features.append('year')

data_features = data[features]

# reset indices for concatenation
embeddings_df.reset_index(drop=True, inplace=True)
data_features.reset_index(drop=True, inplace=True)

# ensure the number of rows matches
if embeddings_df.shape[0] != data_features.shape[0]:
    raise ValueError("Embeddings and data_features do not have the same number of rows.")

X = pd.concat([embeddings_df, data_features], axis=1)

40548


,0,1,2,3,4,5,6,7,8,9,...,year,klima_count,erwärmung_count,treibhaus_count,co2_count,kohle_count,energiewende_count,verkehrswende_count,fridays for future_count,extinction rebellion_count
0,0.400948,-0.158607,-1.367338,0.438427,-0.706825,-0.861589,-0.030629,-0.164405,-0.408176,0.933288,...,2016,1,0,0,0,0,0,0,0,0
1,0.891384,-0.632991,-0.880238,0.297829,-0.903643,-0.788855,-0.148777,0.342639,-0.446057,-0.067019,...,2016,1,0,0,0,0,0,0,0,0
2,0.426084,-0.058124,-0.987131,0.440454,-0.965592,-0.471060,-0.051625,-0.114570,-0.546721,0.317033,...,2016,3,0,0,0,0,0,0,0,0
3,1.088818,-0.174095,-0.696190,0.270226,-0.181869,-0.438196,-0.053129,0.308293,-0.373701,0.614411,...,2016,1,0,0,0,0,0,0,0,0
4,1.215466,0.092294,-0.486537,0.899592,-0.647790,-0.720263,-0.489639,0.342863,-0.194304,0.588071,...,2016,1,0,0,0,0,0,0,0,0


In [39]:
##### STEP 3: load and set up model for inference
xgb_model = xgb.XGBClassifier()
xgb_model.load_model('xgb_model.json')

preds = xgb_model.predict(X) #make predictions

In [ ]:
# concatenate data with predicted label and save
preds_df = pd.DataFrame(preds, columns=['pred_y'])

preds_df.reset_index(drop=True, inplace=True)
data.reset_index(drop=True, inplace=True)

result = pd.concat([data, preds_df], axis=1)
result.to_csv('texts_classified.csv')

#### Viewing results

In [41]:
result.groupby('pred_y').count()

,context,id,author,date,text,text_tokenised,token_count,klima_count,erwärmung_count,treibhaus_count,co2_count,kohle_count,energiewende_count,verkehrswende_count,fridays for future_count,extinction rebellion_count,year
pred_y,,,,,,,,,,,,,,,,,
0,31803,31803,31666,31803,31803,31803,31803,31803,31803,31803,31803,31803,31803,31803,31803,31803,31803
1,8745,8745,8661,8745,8745,8745,8745,8745,8745,8745,8745,8745,8745,8745,8745,8745,8745


#### Saving only texts classified as topic relevant

In [ ]:
result[result['pred_y']==1].to_csv('texts_climate.csv')